# Per-Visit LOWESS Exploration of Fetal Weight Trajectories

Motivation (Dr. Sinha): the regression of $Y_{ij}(t)$ on $t$ may differ substantially across the four measurement phases, so fit **four separate models** rather than one pooled model. These plots are the diagnostic that supports (or refutes) that choice.

Notation: $i$ = subject, $j = 1,2,3,4$ = measurement occasion.

**Tasks**
1. $Y_{ij}(t_{ij})$ vs $t_{ij}$ with LOWESS, one figure per $j$.
2. $Y_{ij}(t_{ij}) - \bar{y}_j$ vs $t_{ij}$ with LOWESS, one figure per $j$, where $\bar{y}_j$ is the sample mean at visit $j$.
3. Both of the above restricted to subjects in the top 15th percentile of birthweight.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

DATA = 'data/longitudinal_data_processed.csv'
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)
FRAC = 0.5   # LOWESS bandwidth
NIT  = 3     # robustness iterations
TRIM = 1.0   # drop t outside [TRIM, 100-TRIM] pctile within each visit; set 0 to disable

## Load and index visits

Every subject has exactly 4 EFW records, so $j$ is just the within-subject rank of `time`.

In [ ]:
df = pd.read_csv(DATA).sort_values(['id', 'time']).reset_index(drop=True)
df['j'] = df.groupby('id').cumcount() + 1

assert set(df.groupby('id').size().unique()) == {4}, 'ragged visit counts'
print(f"{df.id.nunique()} subjects, {len(df)} records")

# visit-level summary: this table is the quantitative version of the 4-model argument
summ = (df.groupby('j')
          .agg(n=('efw', 'size'),
               t_min=('time', 'min'), t_max=('time', 'max'),
               t_sd=('time', 'std'),
               ybar=('efw', 'mean'), y_sd=('efw', 'std'))
          .round(1))
summ['ols_slope'] = [np.polyfit(g.time, g.efw, 1)[0].round(2)
                     for _, g in df.groupby('j')]
summ['cv'] = (summ.y_sd / summ.ybar).round(3)
summ

## Plotting helpers

`TRIM` matters. At the extremes of $t$ within a visit there are one or two isolated observations, and LOWESS there is effectively fitting a line through a single point — the untrimmed curve at visit 2 swings several hundred grams off one record at $t=228$. Excluded points are still drawn (grey ×) so nothing is hidden. Set `TRIM = 0` to reproduce the raw behaviour.

In [ ]:
PHASE = {1: '~17 wk', 2: '~25 wk', 3: '~33 wk', 4: '~37 wk'}


def panel(ax, t, y, j, centered):
    ax.scatter(t, y, s=9, alpha=.30, color='#4878A8', edgecolors='none',
               label=f'observed (n={len(t)})')

    if TRIM > 0:
        lo, hi = np.percentile(t, [TRIM, 100 - TRIM])
        m = (t >= lo) & (t <= hi)
        lab = f'LOWESS (frac={FRAC}, {TRIM}-{100-TRIM} pctile of t)'
    else:
        m = np.ones(len(t), bool)
        lab = f'LOWESS (frac={FRAC}, untrimmed)'

    sm = lowess(y[m], t[m], frac=FRAC, it=NIT, return_sorted=True)
    ax.plot(sm[:, 0], sm[:, 1], color='#C44E52', lw=2.2, label=lab)

    if (~m).sum():
        ax.scatter(t[~m], y[~m], s=18, color='#999', marker='x', lw=.9,
                   label=f'excluded from fit ({(~m).sum()})')

    if centered:
        ax.axhline(0, color='#555', lw=.8, ls='--', zorder=0)
        ax.set_ylabel(r'$Y_{ij}(t_{ij}) - \bar{y}_j$  (g)')
    else:
        ax.set_ylabel(r'$Y_{ij}(t_{ij})$  (g)')

    ax.set_xlabel(r'$t_{ij}$  (gestational days)')
    ax.set_title(f'Visit j = {j}  ({PHASE[j]})', fontsize=11)
    ax.legend(fontsize=8, framealpha=.9)
    ax.grid(alpha=.25, lw=.5)


def make_set(data, centered, suptitle, save_prefix=None):
    """Four standalone figures (as requested) plus a 2x2 composite for review."""
    for j in range(1, 5):
        d = data[data.j == j]
        y = d.efw.values - (d.efw.values.mean() if centered else 0)
        fig, ax = plt.subplots(figsize=(7, 5))
        panel(ax, d.time.values, y, j, centered)
        fig.tight_layout()
        if save_prefix:
            fig.savefig(f'{save_prefix}_visit{j}.png', dpi=160)
        plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))
    for j in range(1, 5):
        d = data[data.j == j]
        y = d.efw.values - (d.efw.values.mean() if centered else 0)
        panel(axes.flat[j - 1], d.time.values, y, j, centered)
    fig.suptitle(suptitle, fontsize=13, y=.995)
    fig.tight_layout()
    if save_prefix:
        fig.savefig(f'{save_prefix}_composite.png', dpi=150)
    plt.show()

## Task 1 — $Y_{ij}(t_{ij})$ vs $t_{ij}$, all subjects

In [ ]:
make_set(df, centered=False,
         suptitle='Task 1 - Fetal weight vs gestational day, by visit (all subjects)',
         save_prefix=f'{FIG_DIR}/task1_raw')

## Task 2 — $Y_{ij}(t_{ij}) - \bar{y}_j$ vs $t_{ij}$, all subjects

Note that subtracting $\bar{y}_j$ is a pure vertical translation within each panel, so the LOWESS **shape and slope are identical to Task 1** — only the $y$-axis origin moves. What this buys is a common scale: the four panels become directly comparable in grams of deviation, which makes the widening dispersion across $j$ readable at a glance.

In [ ]:
make_set(df, centered=True,
         suptitle=r'Task 2 - Visit-centered fetal weight $Y_{ij}-\bar{y}_j$ (all subjects)',
         save_prefix=f'{FIG_DIR}/task2_centered')

## Task 3 — restricted to top 15th percentile of birthweight

Birthweight is constant within subject, so the percentile is taken over subjects, not over records.

In [ ]:
bw_subj = df.groupby('id')['bw'].first()
thresh = bw_subj.quantile(0.85)
top_ids = bw_subj[bw_subj >= thresh].index
top = df[df.id.isin(top_ids)]

print(f'85th pctile birthweight : {thresh:.0f} g')
print(f'subjects retained       : {top.id.nunique()} '
      f'({100*top.id.nunique()/df.id.nunique():.1f}%)')

In [ ]:
make_set(top, centered=False,
         suptitle=f'Task 3a - Fetal weight vs gestational day, top 15% birthweight '
                  f'(bw >= {thresh:.0f} g, n={top.id.nunique()})',
         save_prefix=f'{FIG_DIR}/task3a_raw_top15')

In [ ]:
make_set(top, centered=True,
         suptitle=r'Task 3b - Visit-centered $Y_{ij}-\bar{y}_j$, top 15% birthweight '
                  f'(bw >= {thresh:.0f} g, n={top.id.nunique()})',
         save_prefix=f'{FIG_DIR}/task3b_centered_top15')

## Side-by-side: all subjects vs top 15%

In [ ]:
rows = []
for lbl, d in (('all', df), ('top15', top)):
    for j in range(1, 5):
        g = d[d.j == j]
        rows.append(dict(cohort=lbl, j=j, n=len(g),
                         ybar=round(g.efw.mean(), 1),
                         y_sd=round(g.efw.std(), 1),
                         cv=round(g.efw.std() / g.efw.mean(), 3),
                         ols_slope=round(np.polyfit(g.time, g.efw, 1)[0], 2)))
comp = pd.DataFrame(rows).pivot(index='j', columns='cohort')
comp

## Read-out

Fill in after inspecting the figures. Points the plots should be interrogated on:

- **Does the slope differ enough across $j$ to justify 4 models?** Compare `ols_slope` across visits and check whether the LOWESS within each panel is close to linear.
- **Is dispersion heteroscedastic across $j$?** If `y_sd` grows with $j$ but `cv` is roughly flat, that argues for modelling on a log or proportional scale rather than four unrelated additive models.
- **Does the top-15% cohort differ in slope, or only in intercept?** A pure intercept shift means the growth *process* is common and birthweight extremity is a level effect; a slope difference means the trajectory shape itself differs, which is the more interesting finding for LGA/macrosomia prediction.